In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from simulation.rules import *
from simulation.simulate import *
from config import *
import random

In [2]:
pbp_df = pd.read_parquet(DATA_DIR/PBP_FILE)
drive_list = pd.read_csv(DATA_DIR/DRIVE_FILE)
ko_list = pd.read_csv(DATA_DIR/KO_FILE)

In [11]:
def get_ruleset(season, week):
    is_playoff = week > 18
    if season >= 2025:
        return '2025'
    elif season == 2024:
        return '2024'  # dynamic kickoff, old OT rules
    elif season >= 2022 and is_playoff:
        return '2025'  # playoff OT same as 2025 rules
    elif season >= 2017:
        return '2017-2023'  # 35-yard kickoff
    elif season == 2016:
        return '2016'  # 25-yard touchback
    elif season >= 2012:
        return '2012-2015'  # FG-match era, 15-min OT
    else:
        return 'Pre 2012'  # pure sudden death

In [12]:
def aggregate_overtime_games(pbp: pd.DataFrame) -> pd.DataFrame:
    ot_plays = pbp[pbp['game_half'] == 'Overtime'].copy()

    results = []

    for game_id, game in ot_plays.groupby('game_id'):
        game = game.sort_values('play_id')

        first_play = game.iloc[0]
        season = first_play['season']
        week = first_play['week']
        receiving_team = first_play['posteam']

        home = first_play['home_team']
        away = first_play['away_team']
        kicking_team = away if receiving_team == home else home

        last_play = game.iloc[-1]
        home_score = last_play['total_home_score']
        away_score = last_play['total_away_score']

        if home_score > away_score:
            winner = home
        elif away_score > home_score:
            winner = away
        else:
            winner = 'TIE'

        results.append({
            'game_id': game_id,
            'season': season,
            'week': week,
            'ruleset': get_ruleset(season, week),
            'kicking_team': kicking_team,
            'receiving_team': receiving_team,
            'winner': winner,
            'kicking_team_won': winner == kicking_team,
            'receiving_team_won': winner == receiving_team,
            'tie': winner == 'TIE',
        })

    return pd.DataFrame(results).sort_values(['season', 'week']).reset_index(drop=True)

In [13]:
hist_df = aggregate_overtime_games(pbp_df)

In [14]:
hist_ot_results = hist_df.groupby(['ruleset']).agg(
    games=('game_id', 'count'),
    receiving_pct=('receiving_team_won', 'mean'),
    kicking_pct=('kicking_team_won', 'mean'),
    tie_pct=('tie', 'mean'),
    year = ('season', 'max')
).sort_values(by = 'year').reset_index()

hist_ot_results['receiving_team_adv'] = hist_ot_results['receiving_pct']-hist_ot_results['kicking_pct']

hist_ot_results = hist_ot_results.rename(columns={
    'receiving_pct': '% Receiving Team Won',
    'kicking_pct': '% Kicking Team Won',
    'tie_pct': '% Tie',
    'receiving_team_adv': 'Receiving Team Advantage',
})[['ruleset','games', '% Receiving Team Won', '% Kicking Team Won', '% Tie', 'Receiving Team Advantage']]

hist_ot_results


,ruleset,games,% Receiving Team Won,% Kicking Team Won,% Tie,Receiving Team Advantage
0,Pre 2012,191,0.554974,0.434555,0.010471,0.120419
1,2012-2015,73,0.506849,0.438356,0.054795,0.068493
2,2016,14,0.428571,0.428571,0.142857,0.000000
3,2017-2023,108,0.537037,0.388889,0.074074,0.148148
4,2024,16,0.750000,0.187500,0.062500,0.562500
5,2025,17,0.529412,0.411765,0.058824,0.117647


In [15]:
sim_results = pd.read_csv(OUTPUT_DIR/OT_RESULTS).rename(columns={
            'Receiving Team':'% Receiving Team Won',
            'Kicking Team':'% Kicking Team Won',
            'TIE': '% Tie'
        })

sim_results['ruleset'] = sim_results['season'].apply(lambda x: get_ruleset(x, 1))
merged = pd.merge(hist_ot_results, sim_results, on='ruleset', suffixes=[' Historical', ' Modeled'])
merged = merged.rename(columns = {
    'games Historical': 'Games Historical',
    'games Modeled': 'Games Modeled','season':'Season Modeled'})
merged.to_csv(ASSET_DIR/HIST_COMPARISON, index = False)

In [16]:
merged

,ruleset,Games Historical,% Receiving Team Won Historical,% Kicking Team Won Historical,% Tie Historical,Receiving Team Advantage Historical,Season Modeled,Games Modeled,% Receiving Team Won Modeled,% Kicking Team Won Modeled,% Tie Modeled,Receiving Team Advantage Modeled
0,Pre 2012,191,0.554974,0.434555,0.010471,0.120419,2011,10000,0.5328,0.4172,0.0500,0.1156
1,2012-2015,73,0.506849,0.438356,0.054795,0.068493,2013,10000,0.4834,0.4576,0.0590,0.0258
2,2016,14,0.428571,0.428571,0.142857,0.000000,2016,10000,0.4838,0.4549,0.0613,0.0289
3,2017-2023,108,0.537037,0.388889,0.074074,0.148148,2021,10000,0.4798,0.3925,0.1277,0.0873
4,2024,16,0.750000,0.187500,0.062500,0.562500,2024,10000,0.5344,0.3672,0.0984,0.1672
5,2025,17,0.529412,0.411765,0.058824,0.117647,2025,10000,0.4702,0.3938,0.1360,0.0764
